In [ ]:
# A/B Test Analysis Framework
# Author: Francesc Cebrián
# Portfolio Project: Statistical validation for product experiments

import random
import matplotlib.pyplot as plt
import numpy as np

# ============================================
# BUSINESS CONTEXT
# ============================================
"""
BUSINESS PROBLEM:
Product teams at SaaS companies struggle to measure experiment impact accurately.
Poor analysis leads to:
- Launching features that don't improve metrics
- Missing opportunities worth €50K+/month
- Making decisions without statistical confidence

SOLUTION:s
Automated A/B test analysis with:
- Statistical validation (sample size, confidence)
- Revenue impact calculation
- Clear go/no-go recommendations
"""

# ============================================
# 1. DATA GENERATION (Simulating real test data)
# ============================================

def generate_ab_test_data(n_control=1000, n_treatment=1000, baseline_rate=0.10, uplift=0.03):
    """
    Simulate A/B test data for demonstration
    
    In production, you'd load this from your analytics database:
    - Google Analytics
    - Mixpanel
    - Amplitude
    - Custom data warehouse
    """
    random.seed(42)
    
    # Control group (current version)
    control_conversions = sum(1 for _ in range(n_control) if random.random() < baseline_rate)
    
    # Treatment group (new version with expected uplift)
    treatment_rate = baseline_rate + uplift
    treatment_conversions = sum(1 for _ in range(n_treatment) if random.random() < treatment_rate)
    
    return {
        "control": {"users": n_control, "conversions": control_conversions},
        "treatment": {"users": n_treatment, "conversions": treatment_conversions}
    }

# Generate sample data
test_data = generate_ab_test_data(n_control=2000, n_treatment=2000, baseline_rate=0.10, uplift=0.03)

print("=== A/B TEST DATA ===")
print(f"Control Group: {test_data['control']['users']} users, {test_data['control']['conversions']} conversions")
print(f"Treatment Group: {test_data['treatment']['users']} users, {test_data['treatment']['conversions']} conversions")

# ============================================
# 2. STATISTICAL ANALYSIS FUNCTION
# ============================================

def analyze_ab_test(control_users, control_conversions, 
                    treatment_users, treatment_conversions,
                    revenue_per_conversion=50):
    """
    Complete A/B test analysis with statistical validation
    
    Args:
        control_users (int): Number of users in control group
        control_conversions (int): Number of conversions in control
        treatment_users (int): Number of users in treatment group  
        treatment_conversions (int): Number of conversions in treatment
        revenue_per_conversion (float): Average revenue per conversion (€)
        
    Returns:
        dict: Complete analysis with metrics and recommendation
    """
    
    # Calculate conversion rates
    control_rate = control_conversions / control_users
    treatment_rate = treatment_conversions / treatment_users
    
    # Calculate uplift
    absolute_uplift = treatment_rate - control_rate
    relative_uplift_pct = (absolute_uplift / control_rate) * 100
    
    # Statistical significance (simplified z-test)
    pooled_rate = (control_conversions + treatment_conversions) / (control_users + treatment_users)
    se = np.sqrt(pooled_rate * (1 - pooled_rate) * (1/control_users + 1/treatment_users))
    z_score = absolute_uplift / se
    
    # Confidence level (simplified)
    is_significant = abs(z_score) > 1.96  # 95% confidence
    
    # Business impact
    monthly_users = 10000  # Assumption: 10K users/month
    additional_conversions = monthly_users * absolute_uplift
    monthly_revenue_impact = additional_conversions * revenue_per_conversion
    annual_revenue_impact = monthly_revenue_impact * 12
    
    # Recommendation
    if is_significant and relative_uplift_pct > 5:
        recommendation = "SHIP IT - Strong positive impact"
        confidence = "High"
    elif is_significant and relative_uplift_pct > 0:
        recommendation = "SHIP IT - Positive impact confirmed"
        confidence = "Medium"
    elif not is_significant:
        recommendation = "INCONCLUSIVE - Need more data"
        confidence = "Low"
    else:
        recommendation = "DO NOT SHIP - Negative impact"
        confidence = "High"
    
    return {
        "control_rate": control_rate,
        "treatment_rate": treatment_rate,
        "absolute_uplift": absolute_uplift,
        "relative_uplift_pct": relative_uplift_pct,
        "z_score": z_score,
        "is_significant": is_significant,
        "confidence": confidence,
        "monthly_revenue_impact": monthly_revenue_impact,
        "annual_revenue_impact": annual_revenue_impact,
        "recommendation": recommendation
    }

# ============================================
# 3. RUN ANALYSIS
# ============================================

results = analyze_ab_test(
    control_users=test_data['control']['users'],
    control_conversions=test_data['control']['conversions'],
    treatment_users=test_data['treatment']['users'],
    treatment_conversions=test_data['treatment']['conversions'],
    revenue_per_conversion=50
)

# ============================================
# 4. RESULTS REPORT (Stakeholder-ready format)
# ============================================

print("\n" + "="*60)
print("A/B TEST RESULTS - EXECUTIVE SUMMARY")
print("="*60)

print(f"\n📊 CONVERSION RATES:")
print(f"   Control:   {results['control_rate']:.2%} ({test_data['control']['conversions']}/{test_data['control']['users']} users)")
print(f"   Treatment: {results['treatment_rate']:.2%} ({test_data['treatment']['conversions']}/{test_data['treatment']['users']} users)")

print(f"\n📈 PERFORMANCE:")
print(f"   Absolute Uplift: {results['absolute_uplift']:.2%}")
print(f"   Relative Uplift: {results['relative_uplift_pct']:+.1f}%")

print(f"\n🔬 STATISTICAL VALIDATION:")
print(f"   Z-Score: {results['z_score']:.2f}")
print(f"   Significant: {'✅ YES' if results['is_significant'] else '❌ NO'}")
print(f"   Confidence: {results['confidence']}")

print(f"\n💰 BUSINESS IMPACT (10K users/month):")
print(f"   Monthly Revenue: €{results['monthly_revenue_impact']:,.0f}")
print(f"   Annual Revenue:  €{results['annual_revenue_impact']:,.0f}")

print(f"\n🎯 RECOMMENDATION:")
print(f"   {results['recommendation']}")

print("="*60)

# ============================================
# 5. VISUALIZATION
# ============================================

def plot_ab_test_results(results, test_data):
    """
    Create professional visualization for stakeholders
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Chart 1: Conversion Rate Comparison
    groups = ['Control', 'Treatment']
    rates = [results['control_rate'] * 100, results['treatment_rate'] * 100]
    colors = ['#3498db', '#2ecc71']
    
    bars = ax1.bar(groups, rates, color=colors, alpha=0.8, edgecolor='black')
    ax1.set_ylabel('Conversion Rate (%)', fontsize=12)
    ax1.set_title('A/B Test: Conversion Rate Comparison', fontsize=14, fontweight='bold')
    ax1.set_ylim(0, max(rates) * 1.3)
    
    # Add value labels on bars
    for bar, rate in zip(bars, rates):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{rate:.2f}%',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    # Add uplift annotation
    if results['relative_uplift_pct'] > 0:
        ax1.annotate(f"+{results['relative_uplift_pct']:.1f}% uplift",
                    xy=(1, results['treatment_rate'] * 100),
                    xytext=(0.5, max(rates) * 1.15),
                    arrowprops=dict(arrowstyle='->', color='green', lw=2),
                    fontsize=12, color='green', fontweight='bold')
    
    # Chart 2: Revenue Impact
    impact_types = ['Monthly', 'Annual']
    revenues = [results['monthly_revenue_impact'], results['annual_revenue_impact']]
    
    bars2 = ax2.bar(impact_types, revenues, color=['#f39c12', '#e74c3c'], alpha=0.8, edgecolor='black')
    ax2.set_ylabel('Revenue Impact (€)', fontsize=12)
    ax2.set_title('Projected Revenue Impact', fontsize=14, fontweight='bold')
    ax2.ticklabel_format(style='plain', axis='y')
    
    # Add value labels
    for bar, revenue in zip(bars2, revenues):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'€{revenue:,.0f}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('ab_test_results.png', dpi=300, bbox_inches='tight')
    print("\n✅ Visualization saved as 'ab_test_results.png'")
    plt.show()

# Generate visualization
plot_ab_test_results(results, test_data)

# ============================================
# 6. PRODUCTION CONSIDERATIONS
# ============================================

print("\n" + "="*60)
print("📋 NEXT STEPS FOR PRODUCTION:")
print("="*60)
print("""
1. ✅ Sample Size Validation
   - Minimum 1000 users per variant
   - Current: {0} control, {1} treatment ✓

2. ⚠️  Test Duration
   - Recommended: Run for 2 full weeks
   - Account for day-of-week effects
   
3. 🔍 Segment Analysis (Future Enhancement)
   - Break down by: New vs Returning users
   - By: Device type, Geography, Traffic source
   
4. 💡 What to Watch:
   - Novelty effect (first week inflated results)
   - Sample Ratio Mismatch (unequal traffic split)
   - Multiple testing (running too many experiments)
   
5. 🚀 Deployment Plan:
   - Gradual rollout (10% → 50% → 100%)
   - Monitor metrics for 2 weeks post-launch
   - Set up alerts for anomalies
""".format(test_data['control']['users'], test_data['treatment']['users']))

print("="*60)
print("✅ Analysis complete. Ready to present to stakeholders.")
print("="*60)